# GA Grid Benchmark (Fair Initial Population)

This notebook runs GA benchmarks **one (population, generations) combo per cell** so you can run/save incrementally.

- **Fairness**: For each `population_size`, the **same initial population** is reused across all `generations`.
- **Checkpointing**: Each run saves:
  - a PNG figure (fitness + diversity)
  - a JSON file with the run results
  - appends a row to `summary.csv`
  - updates `summary.xlsx`

If a run takes too long, you can stop and resume later—existing outputs will remain on disk.


In [1]:
# --- Imports ---
import sys
from pathlib import Path

# Assume notebook is inside project root or a subfolder
PROJECT_ROOT = Path.cwd().resolve().parent

# If notebook is in a subfolder, uncomment and adjust:
# PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

import os
import time
import json
import copy
import random
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from func.datastruct import Point, Machine
from gen_algo_params import GeneticAlgorithm


Project root: C:\Users\Dave CH\Documents\fyp-main


In [2]:
# --- Helper: run GA and record diversity history (keeps GA class unchanged) ---

def run_ga_with_diversity(
    ga: GeneticAlgorithm,
    initial_population: Optional[List[np.ndarray]] = None,
    local_search_prob: float = 0.1,
    local_search_iters: int = 5,
    print_every: int = 0
) -> Tuple[List[Machine], float, Dict]:

    # Initialize population (shared or new)
    if initial_population is None:
        population = ga.create_initial_population()
    else:
        population = [chrom.copy() for chrom in initial_population]
        if len(population) != ga.population_size:
            raise ValueError(
                f"Initial population size {len(population)} != GA population_size {ga.population_size}"
            )

    best_solution = None
    best_fitness = float("inf")

    ga.best_fitness_history = []
    ga.avg_fitness_history = []
    diversity_history = []

    start_time = time.time()

    for generation in range(ga.generations):
        # Evaluate fitness
        fitness_scores = [ga.fitness_function(ch) for ch in population]

        # Track best solution
        best_idx = int(np.argmin(fitness_scores))
        if fitness_scores[best_idx] < best_fitness:
            best_fitness = float(fitness_scores[best_idx])
            best_solution = population[best_idx].copy()

        # Stats
        finite = [f for f in fitness_scores if f != float("inf")]
        avg_fitness = float(np.mean(finite)) if len(finite) > 0 else float("inf")
        diversity = float(ga.calculate_diversity(population))

        ga.best_fitness_history.append(best_fitness)
        ga.avg_fitness_history.append(avg_fitness)
        diversity_history.append(diversity)

        if print_every and generation % print_every == 0:
            print(
                f"[pop={ga.population_size} gen={ga.generations}] "
                f"Generation {generation}: Best={best_fitness:.3f}, Avg={avg_fitness:.3f}, Div={diversity:.3f}"
            )

        # Create new population
        new_population = []

        # Elitism
        elite_count = int(ga.elitism_rate * ga.population_size)
        elite_indices = np.argsort(fitness_scores)[:elite_count]
        for idx in elite_indices:
            new_population.append(population[idx].copy())

        # Offspring
        while len(new_population) < ga.population_size:
            p1 = ga.tournament_selection(population, fitness_scores)
            p2 = ga.tournament_selection(population, fitness_scores)

            c1, c2 = ga.crossover(p1, p2)
            c1 = ga.mutate(c1)
            c2 = ga.mutate(c2)

            # Local search on some solutions
            if random.random() < local_search_prob:
                c1 = ga.local_search(c1, max_iterations=local_search_iters)

            new_population.extend([c1, c2])

        population = new_population[:ga.population_size]

    end_time = time.time()

    final_machines = ga.decode_chromosome(best_solution)

    results = {
        "population_size": ga.population_size,
        "generations": ga.generations,
        "execution_time": end_time - start_time,
        "best_fitness_history": ga.best_fitness_history,
        "avg_fitness_history": ga.avg_fitness_history,
        "diversity_history": diversity_history,
        "final_fitness": best_fitness,
        "total_distance": ga.calculate_total_distance(final_machines),
    }

    return final_machines, best_fitness, results


In [3]:
# --- Plot helpers ---

def save_run_plots(outdir: Path, results: Dict):
    pop = results["population_size"]
    gens = results["generations"]

    best_hist = results["best_fitness_history"]
    avg_hist = results["avg_fitness_history"]
    div_hist = results["diversity_history"]

    fig = plt.figure(figsize=(12, 5))

    ax1 = fig.add_subplot(1, 2, 1)
    ax1.plot(best_hist, label="Best Fitness")
    ax1.plot(avg_hist, label="Avg Fitness")
    ax1.set_title(f"Fitness Convergence (pop={pop}, gen={gens})")
    ax1.set_xlabel("Generation")
    ax1.set_ylabel("Fitness")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2 = fig.add_subplot(1, 2, 2)
    ax2.plot(div_hist, label="Diversity")
    ax2.set_title(f"Diversity (pop={pop}, gen={gens})")
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Avg pairwise L2 distance")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    fig.tight_layout()
    fig_path = outdir / f"pop_{pop}_gen_{gens}.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def append_summary(out_root: Path, row: Dict):
    """Append a row to summary.csv and also refresh summary.xlsx."""
    csv_path = out_root / "summary.csv"
    xlsx_path = out_root / "summary.xlsx"

    cols = ["population_size","generations","final_fitness","total_distance","execution_time"]

    # append to CSV (create header if missing)
    write_header = not csv_path.exists()
    with open(csv_path, "a", newline="") as f:
        if write_header:
            f.write(",".join(cols) + "\n")
        f.write(
            f"{row['population_size']},{row['generations']},{row['final_fitness']},{row['total_distance']},{row['execution_time']}\n"
        )

    # refresh Excel from CSV (safe, idempotent)
    df = pd.read_csv(csv_path)
    df.to_excel(xlsx_path, index=False)


In [4]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon, FancyArrowPatch

# If your visualize.py is in the project root (same level as func/, gen_algo_params.py)
# you can import it, but we’ll implement a save-only wrapper to avoid plt.show().
# from visualize import visualize_layout  # optional

def save_layout_figure(
    machines,
    robot_position,
    sequence,
    workspace_bounds,
    save_path: str | Path,
    title: str = "Optimized Robotic Workcell Layout",
    dpi: int = 150,
):
    """Save a layout visualization to disk (no plt.show), and close the figure."""
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(1, 1, figsize=(12, 10))

    # Draw workspace bounds
    min_x, max_x, min_y, max_y = workspace_bounds
    workspace_rect = Rectangle(
        (min_x, min_y),
        max_x - min_x,
        max_y - min_y,
        linewidth=2,
        edgecolor="black",
        facecolor="lightgray",
        alpha=0.3,
    )
    ax.add_patch(workspace_rect)

    # Color map for machines
    colors = plt.cm.Set3(np.linspace(0, 1, len(machines)))

    # Draw machines
    for i, machine in enumerate(machines):
        corners = machine.get_corners()
        polygon_coords = [(p.x, p.y) for p in corners]

        polygon = Polygon(
            polygon_coords, closed=True,
            facecolor=colors[i], edgecolor="black", alpha=0.7
        )
        ax.add_patch(polygon)

        # Label machine
        ax.text(
            machine.position.x, machine.position.y, f"M{machine.id}",
            ha="center", va="center", fontweight="bold", fontsize=10
        )

        # Draw access point
        ap = machine.get_access_point_world()
        ax.plot(ap.x, ap.y, "ro", markersize=8)

    # Draw robot position
    ax.plot(robot_position.x, robot_position.y, "bs", markersize=12, label="Robot")

    # Draw sequence path
    path_points = [robot_position]
    for machine_id in sequence:
        machine = next(m for m in machines if m.id == machine_id)
        path_points.append(machine.get_access_point_world())
    path_points.append(robot_position)

    ax.plot(
        [p.x for p in path_points],
        [p.y for p in path_points],
        "r--",
        linewidth=2,
        alpha=0.7,
        label="Robot Path",
    )

    # Add arrows
    for i in range(len(path_points) - 1):
        arrow = FancyArrowPatch(
            (path_points[i].x, path_points[i].y),
            (path_points[i + 1].x, path_points[i + 1].y),
            arrowstyle="->",
            mutation_scale=15,
            color="red",
            lw=2,
            alpha=0.7,
        )
        ax.add_patch(arrow)

    ax.set_xlim(min_x - 2, max_x + 2)
    ax.set_ylim(min_y - 2, max_y + 2)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("X Position")
    ax.set_ylabel("Y Position")

    fig.tight_layout()
    fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

    return str(save_path)

In [5]:
# --- Problem definition (machines/sequence/bounds) ---

machines = [
    Machine(id=1, shape='l_shape', width=5.21, height=4.33,
            access_point=Point(1.48, -1.12),
            l_cutout_width=2.01, l_cutout_height=1.72),

    Machine(id=2, shape='rectangle', width=4.87, height=3.12,
            access_point=Point(0.92, -0.44)),

    Machine(id=3, shape='l_shape', width=4.56, height=5.46,
            access_point=Point(-0.72, 1.89),
            l_cutout_width=1.58, l_cutout_height=2.74),

    Machine(id=4, shape='rectangle', width=3.44, height=2.61,
            access_point=Point(0.13, 0.89)),

    Machine(id=5, shape='l_shape', width=5.94, height=3.77,
            access_point=Point(1.04, -0.88),
            l_cutout_width=2.23, l_cutout_height=1.32),

    Machine(id=6, shape='rectangle', width=4.32, height=2.48,
            access_point=Point(-0.63, -0.41)),

    Machine(id=7, shape='l_shape', width=3.88, height=4.91,
            access_point=Point(0.55, 1.22),
            l_cutout_width=1.25, l_cutout_height=2.01),

    Machine(id=8, shape='rectangle', width=3.27, height=2.74,
            access_point=Point(0.15, -0.52)),
]

sequence = [1, 2, 3, 4, 5, 6, 7, 8]
robot_position = Point(0, 0)
workspace_bounds = (-10, 10, -10, 10)


In [6]:
# --- Grid settings + output directory ---

population_sizes = (100, 200, 500, 1000, 2000)
generations_list = (100, 200, 500, 1000, 2000)

out_root = Path("ga_grid_results_incremental")
out_root.mkdir(parents=True, exist_ok=True)

# Fairness: cache initial population per population size
initial_pop_cache: Dict[int, List[np.ndarray]] = {}

# Base seed for reproducibility
BASE_SEED = 12345

print("Output folder:", out_root.resolve())


Output folder: C:\Users\Dave CH\Documents\fyp-main\tests\ga_grid_results_incremental


In [7]:
# --- Build (or rebuild) the cached initial populations ---
# Run this once before running any combo cells.

for pop in population_sizes:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,  # irrelevant; only creating initial population
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

print("Cached initial populations for pops:", sorted(initial_pop_cache.keys()))


Cached initial populations for pops: [100, 200, 500, 1000, 2000]


In [8]:
# --- RUN: pop=100, gens=100 ---
pop = 100
gens = 100

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_100\pop_100_gen_100.json


In [9]:
# --- RUN: pop=100, gens=200 ---
pop = 100
gens = 200

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_100\pop_100_gen_200.json


In [10]:
# --- RUN: pop=100, gens=500 ---
pop = 100
gens = 500

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_100\pop_100_gen_500.json


In [11]:
# --- RUN: pop=100, gens=1000 ---
pop = 100
gens = 1000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_100\pop_100_gen_1000.json


In [12]:
# --- RUN: pop=100, gens=2000 ---
pop = 100
gens = 2000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_100\pop_100_gen_2000.json


In [13]:
# --- RUN: pop=200, gens=100 ---
pop = 200
gens = 100

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_200\pop_200_gen_100.json


In [14]:
# --- RUN: pop=200, gens=200 ---
pop = 200
gens = 200

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_200\pop_200_gen_200.json


In [15]:
# --- RUN: pop=200, gens=500 ---
pop = 200
gens = 500

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_200\pop_200_gen_500.json


In [16]:
# --- RUN: pop=200, gens=1000 ---
pop = 200
gens = 1000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_200\pop_200_gen_1000.json


In [17]:
# --- RUN: pop=200, gens=2000 ---
pop = 200
gens = 2000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_200\pop_200_gen_2000.json


In [18]:
# --- RUN: pop=500, gens=100 ---
pop = 500
gens = 100

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_500\pop_500_gen_100.json


In [19]:
# --- RUN: pop=500, gens=200 ---
pop = 500
gens = 200

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_500\pop_500_gen_200.json


In [20]:
# --- RUN: pop=500, gens=500 ---
pop = 500
gens = 500

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_500\pop_500_gen_500.json


In [21]:
# --- RUN: pop=500, gens=1000 ---
pop = 500
gens = 1000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_500\pop_500_gen_1000.json


In [22]:
# --- RUN: pop=500, gens=2000 ---
pop = 500
gens = 2000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_500\pop_500_gen_2000.json


In [23]:
# --- RUN: pop=1000, gens=100 ---
pop = 1000
gens = 100

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_1000\pop_1000_gen_100.json


In [24]:
# --- RUN: pop=1000, gens=200 ---
pop = 1000
gens = 200

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_1000\pop_1000_gen_200.json


In [25]:
# --- RUN: pop=1000, gens=500 ---
pop = 1000
gens = 500

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_1000\pop_1000_gen_500.json


In [26]:
# --- RUN: pop=1000, gens=1000 ---
pop = 1000
gens = 1000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_1000\pop_1000_gen_1000.json


In [27]:
# --- RUN: pop=1000, gens=2000 ---
pop = 1000
gens = 2000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_1000\pop_1000_gen_2000.json


In [28]:
# --- RUN: pop=2000, gens=100 ---
pop = 2000
gens = 100

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_2000\pop_2000_gen_100.json


In [29]:
# --- RUN: pop=2000, gens=200 ---
pop = 2000
gens = 200

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_2000\pop_2000_gen_200.json


In [30]:
# --- RUN: pop=2000, gens=500 ---
pop = 2000
gens = 500

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping:

 ga_grid_results_incremental\pop_2000\pop_2000_gen_500.json


In [31]:
# --- RUN: pop=2000, gens=1000 ---
pop = 2000
gens = 1000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Already exists, skipping: ga_grid_results_incremental\pop_2000\pop_2000_gen_1000.json


In [32]:
# --- RUN: pop=2000, gens=2000 ---
pop = 2000
gens = 2000

# Safety: ensure cache exists (in case kernel restarted)
if pop not in initial_pop_cache:
    seed_for_pop = BASE_SEED + int(pop)
    random.seed(seed_for_pop)
    np.random.seed(seed_for_pop)

    ga_for_init = GeneticAlgorithm(
        machines=machines,
        sequence=sequence,
        robot_position=robot_position,
        workspace_bounds=workspace_bounds,
        population_size=pop,
        generations=1,
    )
    initial_pop_cache[pop] = ga_for_init.create_initial_population()

# Reset RNG so runs are comparable and only differ by how long they run
seed_for_pop = BASE_SEED + int(pop)
random.seed(seed_for_pop)
np.random.seed(seed_for_pop)

ga = GeneticAlgorithm(
    machines=machines,
    sequence=sequence,
    robot_position=robot_position,
    workspace_bounds=workspace_bounds,
    population_size=pop,
    generations=gens,
)

# Reuse the same cached initial population for this pop size
initial_population = [c.copy() for c in initial_pop_cache[pop]]

run_dir = out_root / f"pop_{pop}"
run_dir.mkdir(parents=True, exist_ok=True)

# If already computed, skip unless you want to overwrite
json_path = run_dir / f"pop_{pop}_gen_{gens}.json"
if json_path.exists():
    print("Already exists, skipping:", json_path)
else:
    _, best_fit, res = run_ga_with_diversity(
        ga,
        initial_population=initial_population,
        local_search_prob=0.1,
        local_search_iters=5,
        print_every=0,
    )

    # Save plot + JSON + append summary
    save_run_plots(run_dir, res)
    with open(json_path, "w") as f:
        json.dump(res, f)

    append_summary(out_root, res)

    print(f"Done: pop={pop}, gen={gens} | best={best_fit:.4f} | time={res['execution_time']:.2f}s")

    layout_path = run_dir / f"layout_pop_{pop}_gen_{gens}.png"
    save_layout_figure(
        machines=_,
        robot_position=robot_position,
        sequence=sequence,
        workspace_bounds=workspace_bounds,
        save_path=layout_path,
        title=f"Layout (Population Size = {pop}, No. Generations = {gens}), Best Fitness = {best_fit:.3f}"
    )
    print("Saved layout:", layout_path)


Done: pop=2000, gen=2000 | best=35.9366 | time=25364.26s
Saved layout: ga_grid_results_incremental\pop_2000\layout_pop_2000_gen_2000.png


In [33]:
# --- Post-run: make overlay plots from whatever results exist on disk ---

def load_all_results(out_root: Path) -> List[Dict]:
    results = []
    for pop_dir in sorted(out_root.glob("pop_*")):
        for jp in sorted(pop_dir.glob("pop_*_gen_*.json")):
            with open(jp, "r") as f:
                results.append(json.load(f))
    return results

def save_population_summary_plots(outdir: Path, all_results_for_pop: List[Dict]):
    if not all_results_for_pop:
        return
    pop = all_results_for_pop[0]["population_size"]

    # Best fitness overlay
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1)
    for r in sorted(all_results_for_pop, key=lambda x: x["generations"]):
        ax.plot(r["best_fitness_history"], label=f"gen={r['generations']}")
    ax.set_title(f"Best Fitness vs Generation (pop={pop})")
    ax.set_xlabel("Generation")
    ax.set_ylabel("Best Fitness")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir / f"overlay_bestfitness_pop_{pop}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Diversity overlay
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1)
    for r in sorted(all_results_for_pop, key=lambda x: x["generations"]):
        ax.plot(r["diversity_history"], label=f"gen={r['generations']}")
    ax.set_title(f"Diversity vs Generation (pop={pop})")
    ax.set_xlabel("Generation")
    ax.set_ylabel("Diversity")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(outdir / f"overlay_diversity_pop_{pop}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

all_results = load_all_results(out_root)
print("Loaded runs:", len(all_results))

# per-pop overlays
for pop in population_sizes:
    pop_results = [r for r in all_results if r["population_size"] == pop]
    save_population_summary_plots(out_root, pop_results)

# best by pop (from what exists)
best_by_pop = {}
for pop in population_sizes:
    candidates = [r for r in all_results if r["population_size"] == pop]
    if candidates:
        best_by_pop[pop] = min(candidates, key=lambda x: x["final_fitness"])

print("\n=== Best generations per population size (from completed runs) ===")
for pop in population_sizes:
    if pop in best_by_pop:
        r = best_by_pop[pop]
        print(f"pop={pop}: best_gen={r['generations']} | best_fitness={r['final_fitness']:.4f} | time={r['execution_time']:.2f}s")
    else:
        print(f"pop={pop}: (no runs yet)")


Loaded runs: 25

=== Best generations per population size (from completed runs) ===
pop=100: best_gen=2000 | best_fitness=36.3446 | time=648.30s
pop=200: best_gen=2000 | best_fitness=32.0888 | time=1352.28s
pop=500: best_gen=2000 | best_fitness=33.4592 | time=3874.17s
pop=1000: best_gen=2000 | best_fitness=36.8032 | time=8939.52s
pop=2000: best_gen=2000 | best_fitness=35.9366 | time=25364.26s
